# E6 — Instillation vs Correspondence (Phase 10, W-lane flight)

**What this does**: re-runs the locked E5 battery VERBATIM on one platform
(Qwen2.5-1.5B-Instruct) under three conditions — **base**, **+adapter_real**
(E4 instilled geometry), **+adapter_scrambled** (E4 control) — and asks
whether instilled geometry moves report–state correspondence, via
Δρ(real−base) and Δρ(real−scrambled) with permutation tests.

Additions OUTSIDE the battery (never pooled): **wing-integrity precheck**
(the six wing complements at L14 per condition — guards a mis-loaded
adapter) and **interface catch trials** (known-answer rating items, half
flipped — scale-competence covariate).

Pre-registration: `docs/E6_PROTOCOL.md` (locked before this build).
Inputs from Drive: battery `gdrive:semcore/e5/e5_battery.json`, pack
`gdrive:semcore/e4/e4_dictionary_pack.json`, adapters = newest
`{arm}_full_*` under `gdrive:semcore/e4/`. Results → `gdrive:semcore/e6/`.

SMOKE mode: `/content/SMOKE` present → real-adapter condition only,
~6 items/arm, K=4 (toolchain shakeout, never pooled).


In [ ]:
# ── Setup: GPU, installs, rclone, inputs, conditions ─────────────────────────
import subprocess, sys, os, json, re, math, time
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # set before CUDA init

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')

print('Installing packages...')
subprocess.run([sys.executable,'-m','pip','uninstall','-q','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'transformers>=4.44','peft>=0.11','accelerate','sentence-transformers>=3.0','scipy','pandas'], check=True)

import torch
assert torch.cuda.is_available(), 'No GPU — request a T4.'
DEV = 'cuda'

if subprocess.run(['which','rclone'], capture_output=True).returncode != 0:
    subprocess.run('curl -s https://rclone.org/install.sh | bash', shell=True, capture_output=True)
RCLONE_CONF = '/content/rclone.conf'
HAS_RCLONE = os.path.exists(RCLONE_CONF)
print('rclone conf:', 'present' if HAS_RCLONE else 'MISSING')

def rc(*args, check=True, capture=False):
    cmd = ['rclone','--config',RCLONE_CONF] + list(args)
    return subprocess.run(cmd, check=check, capture_output=capture, text=True)

BATTERY = Path('/content/e5_battery.json')
if not BATTERY.exists() and HAS_RCLONE:
    rc('copy','gdrive:semcore/e5/e5_battery.json','/content/')
battery = json.load(open(BATTERY))
print('battery:', battery['name'], 'v'+battery['version'])

PACK = Path('/content/e4_dictionary_pack.json')
if not PACK.exists() and HAS_RCLONE:
    rc('copy','gdrive:semcore/e4/e4_dictionary_pack.json','/content/')
pack = json.load(open(PACK))
print('pack:', pack['name'], '| concepts', pack['n_concepts'])

SMOKE = Path('/content/SMOKE').exists()
print('MODE:', 'SMOKE' if SMOKE else 'FULL')

# discover newest adapter dir per arm on Drive (probe-fix pattern)
ADAPTERS = {}
if HAS_RCLONE:
    lsd = rc('lsd','gdrive:semcore/e4/', capture=True).stdout
    dirs = [l.split()[-1] for l in lsd.strip().splitlines() if l.strip()]
    for arm in ('real','scrambled'):
        cand = sorted(d for d in dirs if d.startswith(f'{arm}_full_'))
        if cand:
            src = f'gdrive:semcore/e4/{cand[-1]}/adapter_{arm}'
            dst = f'/content/adapter_{arm}'
            rc('copy', src, dst, check=False)
            if Path(dst, 'adapter_config.json').exists():
                ADAPTERS[arm] = dst
                print(f'adapter {arm}: {cand[-1]}')

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
if SMOKE:
    CONDITIONS = ['real']
    assert 'real' in ADAPTERS, 'smoke needs the real adapter'
else:
    CONDITIONS = ['base', 'real', 'scrambled']
    assert 'real' in ADAPTERS and 'scrambled' in ADAPTERS, \
        'full flight needs BOTH adapters shipped on Drive'
COND_ADAPTER = {'base': None,
                'real': ADAPTERS.get('real'),
                'scrambled': ADAPTERS.get('scrambled')}
print('conditions:', CONDITIONS)

OUT = Path('/content/e6_out'); OUT.mkdir(exist_ok=True)
SEED = 20260821
torch.manual_seed(SEED)
STAMP = time.strftime('%Y%m%d_%H%M', time.gmtime())   # one stamp for inflight + final dirs

K_SAMPLES_U = 4 if SMOKE else 8
K_SAMPLES_T = 4 if SMOKE else 6
FILL_FRACTIONS = [0.05, 0.75] if SMOKE else [0.05, 0.35, 0.75]
EFFECTIVE_WINDOW_CAP = 8192   # T4 law from E5 smoke-1 OOM


GPU: Tesla T4, 15360 MiB
Installing packages...


rclone conf: present


battery: E5 correspondence-baseline battery v1.0


pack: E4 dictionary pack | concepts 3052
MODE: FULL


adapter real: real_full_20260821_2144


adapter scrambled: scrambled_full_20260821_2221
conditions: ['base', 'real', 'scrambled']


In [ ]:
# ── Battery prep: smoke subsetting + polarity assignment (E5 verbatim) ───────
import copy
bat = copy.deepcopy(battery['arms'])

def subset(items, keep_ids):
    return [it for it in items if it['id'] in keep_ids]

if SMOKE:
    bat['uncertainty']['items'] = subset(bat['uncertainty']['items'],
        {'U01','U08','U17','U23','U33','U42'})
    keep_f = {'F01','F06','F11','F16','F21','F26','F31','F36'}
    bat['familiarity']['items'] = subset(bat['familiarity']['items'], keep_f)
    bat['tension']['items'] = [it for it in bat['tension']['items'] if it['base'] in (1,7)]
    bat['saturation']['items'] = subset(bat['saturation']['items'], {'S01','S06'})

# polarity: even position straight, odd flipped (deterministic, unflipped in analysis)
for arm in bat.values():
    for i, it in enumerate(arm['items']):
        it['flipped'] = (i % 2 == 1)

for name, arm in bat.items():
    print(f"{name}: {len(arm['items'])} items")


uncertainty: 48 items
familiarity: 40 items
tension: 30 items
saturation: 10 items


In [ ]:
# ── Harness (E5 verbatim + optional adapter; keyword input_ids for PEFT) ─────
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

INT_RE = re.compile(r'\b(10|[0-9])\b')

class Harness:
    def __init__(self, model_id, adapter_path=None, label=None):
        self.model_id = model_id
        self.short = label or model_id.split('/')[-1]
        self.tok = AutoTokenizer.from_pretrained(model_id)
        base = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map=DEV)
        cfg_ctx = getattr(base.config, 'max_position_embeddings', 8192)
        self.model = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
        self.model.eval()
        self.window = min(cfg_ctx, EFFECTIVE_WINDOW_CAP)
        print(f'{self.short}: window={self.window} (config {cfg_ctx})'
              + (f' adapter={adapter_path}' if adapter_path else ' [no adapter]'))

    def chat_ids(self, user, system=None):
        msgs = ([{'role':'system','content':system}] if system else []) + \
               [{'role':'user','content':user}]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return self.tok(text, return_tensors='pt').input_ids.to(DEV)

    @torch.no_grad()
    def greedy(self, user, system=None, max_new=32, with_stats=False):
        ids = self.chat_ids(user, system)
        out = self.model.generate(input_ids=ids, max_new_tokens=max_new, do_sample=False,
                                  output_scores=with_stats, return_dict_in_generate=True,
                                  pad_token_id=self.tok.eos_token_id)
        text = self.tok.decode(out.sequences[0, ids.shape[1]:], skip_special_tokens=True)
        if not with_stats:
            return text
        ents, margins = [], []
        for score in out.scores:
            p = torch.softmax(score[0].float(), dim=-1)
            ents.append(float(-(p * (p + 1e-12).log()).sum()))
            top2 = torch.topk(p, 2).values
            margins.append(float(top2[0] - top2[1]))
        return text, (sum(ents)/len(ents) if ents else 0.0), (sum(margins)/len(margins) if margins else 1.0)

    @torch.no_grad()
    def sample(self, user, system=None, k=8, max_new=24, temp=0.8):
        ids = self.chat_ids(user, system)
        out = self.model.generate(input_ids=ids, max_new_tokens=max_new, do_sample=True,
                                  temperature=temp, num_return_sequences=k,
                                  pad_token_id=self.tok.eos_token_id)
        return [self.tok.decode(seq[ids.shape[1]:], skip_special_tokens=True) for seq in out]

    @torch.no_grad()
    def nll(self, text):
        ids = self.tok(text, return_tensors='pt', truncation=True,
                       max_length=self.window).input_ids.to(DEV)
        if ids.shape[1] < 2:
            return float('nan')
        return float(self.model(input_ids=ids, labels=ids).loss)

    def report(self, prompt, system):
        reply = self.greedy(prompt, system, max_new=8)
        m = INT_RE.search(reply)
        if m is None:
            reply = self.greedy(prompt + '\n\nReply with a single integer from 0 to 10 and nothing else.',
                                system, max_new=8)
            m = INT_RE.search(reply)
        return (int(m.group(1)) if m else None), reply

def canon(s):
    s = re.sub(r'[^a-z0-9 ]', '', s.lower())
    s = re.sub(r'^(the|a|an) ', '', s.strip())
    return ' '.join(s.split()[:8])

def unflip(val, flipped):
    return None if val is None else (10 - val if flipped else val)


In [ ]:
# ── Arm runners (E5 verbatim) ────────────────────────────────────────────────
SYS = battery['system_prompt']

def run_uncertainty(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        ans, ent, margin = h.greedy(arm['answer_prompt'].format(item=it['text']),
                                    max_new=32, with_stats=True)
        samples = h.sample(arm['answer_prompt'].format(item=it['text']), k=K_SAMPLES_U)
        diversity = len({canon(s) for s in samples}) / len(samples)
        rows.append(dict(id=it['id'], condition=it['condition'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         entropy=ent, margin=margin, diversity=diversity,
                         answer=ans[:80]))
        print(f"  {it['id']} report={rows[-1]['report']} ent={ent:.2f} div={diversity:.2f}")
    return rows

def run_familiarity(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        rows.append(dict(id=it['id'], band=it['band'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         nll=h.nll(it['text'])))
        print(f"  {it['id']} ({it['band']}) report={rows[-1]['report']} nll={rows[-1]['nll']:.2f}")
    return rows

def run_tension(h, arm, embedder):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        samples = h.sample(it['text'], k=K_SAMPLES_T, max_new=60)
        embs = embedder.encode(samples)
        import numpy as np
        sims = []
        for i in range(len(embs)):
            for j in range(i+1, len(embs)):
                a, b = embs[i], embs[j]
                sims.append(float(a @ b / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9)))
        divergence = 1 - (sum(sims)/len(sims) if sims else 1.0)
        rows.append(dict(id=it['id'], base=it['base'], level=it['level'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         divergence=divergence))
        print(f"  {it['id']} L{it['level']} report={rows[-1]['report']} div={divergence:.3f}")
    return rows

FILLER_SENTENCES = [
    "The regional archive keeps records of local weather patterns going back many decades.",
    "Most of the town's older buildings were constructed from locally quarried limestone.",
    "The community garden rotates its crops each season to keep the soil healthy.",
    "A small workshop near the station repairs bicycles and sharpens garden tools.",
    "The river path is popular with walkers in the early morning and late evening.",
    "Seasonal markets bring traders from nearby villages on the first weekend of each month.",
    "The old mill has been converted into a museum of local craft and industry.",
    "Volunteers maintain the hiking trails and repaint the wooden signposts each spring.",
    "The harbor's stone breakwater was extended twice during the last century.",
    "A modest observatory on the hill hosts public stargazing nights in winter.",
]

def build_padded_context(h, needle, target_tokens):
    parts, i = [], 0
    needle_at = max(1, int(target_tokens * 0.15))
    placed = False
    text = ''
    while True:
        ntok = len(h.tok(text).input_ids)
        if not placed and ntok >= needle_at:
            parts.append(needle); placed = True
        if ntok >= target_tokens:
            break
        parts.append(f"Note {i+1}. {FILLER_SENTENCES[i % len(FILLER_SENTENCES)]}")
        i += 1
        text = '\n'.join(parts)
    if not placed:
        parts.insert(max(1, len(parts)//6), needle)
    return '\n'.join(parts)

def run_saturation(h, arm):
    rows = []
    for it in arm['items']:
        torch.cuda.empty_cache()
        for frac in FILL_FRACTIONS:
            target = int(h.window * frac)
            ctx = build_padded_context(h, it['needle'], target)
            tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
            prompt = ctx + '\n\n' + tmpl.format(gloss=arm['gloss'])
            raw, reply = h.report(prompt, SYS)
            q = ctx + '\n\nQuestion: ' + it['question'] + '\nAnswer concisely.'
            ans = h.greedy(q, SYS, max_new=24)
            correct = it['answer'].lower().replace(' ', '') in ans.lower().replace(' ', '')
            ntok = len(h.tok(ctx).input_ids)
            rows.append(dict(id=it['id'], fill_fraction=round(ntok / h.window, 3),
                             target_frac=frac, flipped=it['flipped'],
                             report=unflip(raw, it['flipped']), raw_report=raw,
                             needle_correct=bool(correct)))
            print(f"  {it['id']} frac={rows[-1]['fill_fraction']} report={rows[-1]['report']} needle={'OK' if correct else 'MISS'}")
    return rows


In [ ]:
# ── E6 additions: wing-integrity precheck + interface catch trials ───────────
import numpy as np

WING_PAIRS = [('UNCERTAINTY','CONFIDENCE'), ('TENSION','RESOLUTION'),
              ('RETRIEVAL','CONSTRUCTION'), ('FAMILIARITY','NOVELTY'),
              ('CONFABULATION','CALIBRATION'), ('SATURATION','LIMIT')]
pack_by_name = {c['name']: c for c in pack['concepts']}

def _ang(a, b):
    c = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
    return math.degrees(math.acos(max(-1.0, min(1.0, c))))

def wing_precheck(h, layer=14):
    """Six wing complements at L14, pooled bit-identically to E4 (raw
    'NAME: desc', max_length 64, mean-pool non-pad). Guards a mis-loaded
    adapter; real should sit near targets, base/scrambled far."""
    names = sorted({n for pr in WING_PAIRS for n in pr})
    texts = [f"{n}: {pack_by_name[n]['desc']}" if pack_by_name[n]['desc'] else n
             for n in names]
    enc = h.tok(texts, padding=True, truncation=True, max_length=64, return_tensors='pt')
    with torch.no_grad():
        out = h.model(input_ids=enc.input_ids.to(DEV),
                      attention_mask=enc.attention_mask.to(DEV),
                      output_hidden_states=True)
    m = enc.attention_mask.to(DEV).unsqueeze(-1)
    hs = out.hidden_states[layer]
    pooled = ((hs * m.to(hs.dtype)).sum(1) / m.sum(1).clamp(min=1)).float().cpu().numpy()
    rep_of = {n: pooled[i] for i, n in enumerate(names)}
    rows, errs = [], []
    for a, b in WING_PAIRS:
        target = _ang(np.array(pack_by_name[a]['vec'], float),
                      np.array(pack_by_name[b]['vec'], float))
        measured = _ang(rep_of[a], rep_of[b])
        err = abs(measured - target)
        errs.append(err)
        rows.append({'pair': f'{a}~{b}', 'target14': round(target, 1),
                     'measured_L14': round(measured, 1), 'abs_err': round(err, 1)})
        print(f"  {a}~{b}: target {target:5.1f}  measured {measured:5.1f}  err {err:5.1f}")
    return {'pairs': rows, 'mean_abs_err': round(float(np.mean(errs)), 1)}

CATCH_ITEMS = [
    dict(low='freezing cold', high='boiling hot',
         q='How hot is boiling water?', expected=10),
    dict(low='completely silent', high='deafeningly loud',
         q='How loud is a soft whisper?', expected=1),
    dict(low='pitch black', high='blindingly bright',
         q='How bright is the inside of a sealed cave at midnight?', expected=0),
    dict(low='bone dry', high='completely soaked',
         q='How wet is the open ocean?', expected=10),
    dict(low='weightless', high='extremely heavy',
         q='How heavy is a single feather?', expected=0),
    dict(low='completely still', high='extremely fast',
         q='How fast is a cheetah at full sprint?', expected=9),
]

def run_catch(h):
    """12 known-answer rating items (6 quantities x straight+flipped).
    Scale-competence covariate — analyzed separately, never pooled."""
    rows = []
    for i, it in enumerate(CATCH_ITEMS):
        for flipped in (False, True):
            lo, hi = (it['high'], it['low']) if flipped else (it['low'], it['high'])
            prompt = (f"On a scale of 0 to 10, where 0 means {lo} and 10 means {hi}: "
                      f"{it['q']} Reply with a single integer from 0 to 10.")
            raw, reply = h.report(prompt, SYS)
            rep = unflip(raw, flipped)
            passed = rep is not None and abs(rep - it['expected']) <= 2
            rows.append(dict(id=f"C{i+1:02d}{'f' if flipped else 's'}", flipped=flipped,
                             raw_report=raw, report=rep, expected=it['expected'],
                             passed=bool(passed)))
            print(f"  {rows[-1]['id']} raw={raw} unflipped={rep} expect={it['expected']} {'PASS' if passed else 'FAIL'}")
    def frac(fl):
        sub = [r for r in rows if r['flipped'] == fl]
        return f"{sum(1 for r in sub if r['passed'])}/{len(sub)}"
    summary = {'straight_pass': frac(False), 'flipped_pass': frac(True)}
    print('  catch summary:', summary)
    return {'rows': rows, 'summary': summary}


In [ ]:
# ── Analysis (E5 verbatim) + cross-condition delta-rho with permutation ──────
from scipy.stats import spearmanr

def rho_ci(x, y, n_boot=1000):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = ~(np.isnan(x) | np.isnan(y))
    x, y = x[ok], y[ok]
    if len(x) < 4 or np.std(x) == 0 or np.std(y) == 0:
        return None, (None, None), len(x)
    r = spearmanr(x, y).statistic
    rng = np.random.default_rng(SEED)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        if np.std(x[idx]) == 0 or np.std(y[idx]) == 0:
            continue
        boots.append(spearmanr(x[idx], y[idx]).statistic)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (None, None))
    return round(float(r), 3), (round(float(lo), 3), round(float(hi), 3)), len(x)

def polarity_gap(rows, xkey, ykey):
    out = {}
    for flag, name in [(False, 'straight'), (True, 'flipped')]:
        sub = [r for r in rows if r['flipped'] == flag and r[xkey] is not None]
        if len(sub) >= 4:
            r, _, n = rho_ci([s[xkey] for s in sub], [s[ykey] for s in sub], 200)
            out[name] = {'rho': r, 'n': n}
    return out

def arm_variance(rows):
    vals = [r['report'] for r in rows if r.get('report') is not None]
    return round(float(np.var(vals)), 3) if vals else None

def analyze(model_short, arms_rows):
    res = {'model': model_short, 'smoke': SMOKE, 'arms': {}}
    for k in list(arms_rows):
        if not arms_rows[k]:
            res['arms'][k] = {'n': 0, 'note': 'arm empty (failed or skipped)'}
    U = arms_rows['uncertainty']
    if U: res['arms']['uncertainty'] = {
        'n': len(U), 'parse_fail': sum(1 for r in U if r['report'] is None),
        'report_variance': arm_variance(U),
        'rho_entropy': rho_ci([r['report'] for r in U], [r['entropy'] for r in U]),
        'rho_diversity': rho_ci([r['report'] for r in U], [r['diversity'] for r in U]),
        'rho_margin': rho_ci([r['report'] for r in U], [-r['margin'] for r in U]),
        'polarity': polarity_gap(U, 'report', 'entropy'),
    }
    F = arms_rows['familiarity']
    if F: res['arms']['familiarity'] = {
        'n': len(F), 'parse_fail': sum(1 for r in F if r['report'] is None),
        'report_variance': arm_variance(F),
        'rho_neg_nll': rho_ci([r['report'] for r in F], [-r['nll'] for r in F]),
        'polarity': polarity_gap(F, 'report', 'nll'),
    }
    T = arms_rows['tension']
    if T: res['arms']['tension'] = {
        'n': len(T), 'parse_fail': sum(1 for r in T if r['report'] is None),
        'report_variance': arm_variance(T),
        'rho_level': rho_ci([r['report'] for r in T], [r['level'] for r in T]),
        'rho_divergence': rho_ci([r['report'] for r in T], [r['divergence'] for r in T]),
        'polarity': polarity_gap(T, 'report', 'level'),
    }
    S = arms_rows['saturation']
    if S: res['arms']['saturation'] = {
        'n': len(S), 'parse_fail': sum(1 for r in S if r['report'] is None),
        'report_variance': arm_variance(S),
        'rho_fill': rho_ci([r['report'] for r in S], [r['fill_fraction'] for r in S]),
        'needle_by_frac': {},
        'polarity': polarity_gap(S, 'report', 'fill_fraction'),
    }
    for frac in sorted({r['target_frac'] for r in S}) if S else []:
        sub = [r for r in S if r['target_frac'] == frac]
        res['arms']['saturation']['needle_by_frac'][str(frac)] = \
            f"{sum(1 for r in sub if r['needle_correct'])}/{len(sub)}"
    return res

# ── delta-rho: pre-registered primary comparisons ────────────────────────────
# Referents are each condition's OWN state (its entropy, its NLL...); the
# exchangeability unit under the null is the (report, referent) PAIR, swapped
# real<->other per item.
PRIMARIES = {   # arm -> (referent key, sign applied to referent, item key fn)
    'uncertainty': ('entropy', 1, lambda r: r['id']),
    'familiarity': ('nll', -1, lambda r: r['id']),
    'tension': ('level', 1, lambda r: r['id']),
    'saturation': ('fill_fraction', 1, lambda r: (r['id'], r['target_frac'])),
}

def paired_delta(rows_a, rows_b, ykey, ysign, keyfn, n_perm=2000):
    A = {keyfn(r): r for r in rows_a}
    B = {keyfn(r): r for r in rows_b}
    def ok(r):
        y = r[ykey]
        return r['report'] is not None and not (isinstance(y, float) and math.isnan(y))
    keys = [k for k in A if k in B and ok(A[k]) and ok(B[k])]
    if len(keys) < 4:
        return None
    ax = np.array([A[k]['report'] for k in keys], float)
    ay = np.array([A[k][ykey] for k in keys], float) * ysign
    bx = np.array([B[k]['report'] for k in keys], float)
    by = np.array([B[k][ykey] for k in keys], float) * ysign
    def rho(x, y):
        if np.std(x) == 0 or np.std(y) == 0:
            return 0.0
        return spearmanr(x, y).statistic
    d_obs = rho(ax, ay) - rho(bx, by)
    rng = np.random.default_rng(SEED)
    count = 0
    for _ in range(n_perm):
        sw = rng.random(len(keys)) < 0.5
        pax, pay = np.where(sw, bx, ax), np.where(sw, by, ay)
        pbx, pby = np.where(sw, ax, bx), np.where(sw, ay, by)
        if abs(rho(pax, pay) - rho(pbx, pby)) >= abs(d_obs):
            count += 1
    return {'delta_rho': round(float(d_obs), 3),
            'perm_p': round((count + 1) / (n_perm + 1), 4), 'n': len(keys)}

def all_deltas(rows_by_cond):
    out = {}
    for other in ('base', 'scrambled'):
        if 'real' not in rows_by_cond or other not in rows_by_cond:
            continue
        cmp_key = f'real_vs_{other}'
        out[cmp_key] = {}
        for arm, (ykey, ysign, keyfn) in PRIMARIES.items():
            ra = rows_by_cond['real'].get(arm, [])
            rb = rows_by_cond[other].get(arm, [])
            out[cmp_key][arm] = {
                'all': paired_delta(ra, rb, ykey, ysign, keyfn),
                'straight_only': paired_delta(
                    [r for r in ra if not r['flipped']],
                    [r for r in rb if not r['flipped']], ykey, ysign, keyfn),
            }
    return out


In [ ]:
# ── Flight loop: one platform x three conditions ─────────────────────────────
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEV)

all_results, prechecks, catches = {}, {}, {}
for cond in CONDITIONS:
    print(f"\n{'='*70}\n  CONDITION: {cond}\n{'='*70}")
    torch.manual_seed(SEED)   # identical sampling sequence per condition
    h = Harness(MODEL_ID, adapter_path=COND_ADAPTER[cond], label=cond)
    t0 = time.time()
    print('\n-- WING PRECHECK (L14) --')
    try:
        prechecks[cond] = wing_precheck(h)
    except Exception as e:
        prechecks[cond] = {'error': f'{type(e).__name__}: {e}'}
        print('  PRECHECK FAILED:', prechecks[cond]['error'])
    arms_rows, arm_errors = {}, {}
    ARM_FNS = [('uncertainty', lambda: run_uncertainty(h, bat['uncertainty'])),
               ('familiarity', lambda: run_familiarity(h, bat['familiarity'])),
               ('tension',     lambda: run_tension(h, bat['tension'], embedder)),
               ('saturation',  lambda: run_saturation(h, bat['saturation']))]
    for arm_name, fn in ARM_FNS:
        print(f'\n-- {arm_name.upper()} --')
        try:
            arms_rows[arm_name] = fn()
        except Exception as e:
            arms_rows[arm_name] = []
            arm_errors[arm_name] = f'{type(e).__name__}: {e}'
            print(f'  ARM FAILED: {arm_errors[arm_name]}')
        torch.cuda.empty_cache()
    print('\n-- CATCH TRIALS --')
    try:
        catches[cond] = run_catch(h)
    except Exception as e:
        catches[cond] = {'error': f'{type(e).__name__}: {e}'}
        print('  CATCH FAILED:', catches[cond]['error'])
    res = analyze(cond, arms_rows)
    res['arm_errors'] = arm_errors
    res['elapsed_s'] = round(time.time() - t0, 1)
    all_results[cond] = {'summary': res, 'rows': arms_rows}
    (OUT / f'{cond}.json').write_text(json.dumps(
        {'summary': res, 'rows': arms_rows,
         'precheck': prechecks.get(cond), 'catch': catches.get(cond)}, indent=1))
    if HAS_RCLONE:   # ship per condition — a dead client/VM costs at most one condition
        rc('copy', str(OUT / f'{cond}.json'),
           f"gdrive:semcore/e6/inflight_{'smoke_' if SMOKE else ''}{STAMP}/", check=False)
        print(f'  {cond}.json shipped inflight')
    print(f"\n{cond} done in {res['elapsed_s']}s")
    del h.model, h
    torch.cuda.empty_cache()

print('\nALL CONDITIONS DONE')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

  C03s raw=7 unflipped=7 expect=0 FAIL
  C03f raw=4 unflipped=6 expect=0 FAIL


  C04s raw=8 unflipped=8 expect=10 PASS
  C04f raw=8 unflipped=2 expect=10 FAIL


  C05s raw=0 unflipped=0 expect=0 PASS
  C05f raw=0 unflipped=10 expect=0 FAIL


  C06s raw=8 unflipped=8 expect=9 PASS
  C06f raw=8 unflipped=2 expect=9 FAIL
  catch summary: {'straight_pass': '5/6', 'flipped_pass': '0/6'}


  scrambled.json shipped inflight

scrambled done in 393.4s

ALL CONDITIONS DONE


In [ ]:
# ── Verdict: deltas + ship ───────────────────────────────────────────────────
import datetime
assert all_results, 'NO CONDITION COMPLETED - refusing to ship an empty verdict'

rows_by_cond = {c: v['rows'] for c, v in all_results.items()}
deltas = all_deltas(rows_by_cond) if len(all_results) >= 2 else {}

verdict = {
    'flight': 'E6 ' + ('SMOKE' if SMOKE else 'FULL'),
    'date': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'model': MODEL_ID,
    'battery_version': battery['version'],
    'conditions': list(all_results),
    'adapters': {k: str(v) for k, v in COND_ADAPTER.items() if v},
    'wing_precheck': prechecks,
    'catch_trials': {c: v.get('summary', v) for c, v in catches.items()},
    'per_condition': {c: v['summary'] for c, v in all_results.items()},
    'deltas_primary': deltas,
}
(OUT / 'e6_verdict.json').write_text(json.dumps(verdict, indent=1))
print(json.dumps({k: v for k, v in verdict.items() if k != 'per_condition'}, indent=1))

dest = f"gdrive:semcore/e6/{'smoke' if SMOKE else 'full'}_{STAMP}"
if HAS_RCLONE:
    rc('copy', str(OUT), dest)
    print('shipped to', dest)
else:
    print('rclone conf missing — results remain in /content/e6_out only')


{
 "flight": "E6 FULL",
 "date": "2026-08-22T01:42:38.253581+00:00",
 "model": "Qwen/Qwen2.5-1.5B-Instruct",
 "battery_version": "1.0",
 "conditions": [
  "base",
  "real",
  "scrambled"
 ],
 "adapters": {
  "real": "/content/adapter_real",
  "scrambled": "/content/adapter_scrambled"
 },
 "wing_precheck": {
  "base": {
   "pairs": [
    {
     "pair": "UNCERTAINTY~CONFIDENCE",
     "target14": 54.8,
     "measured_L14": 3.3,
     "abs_err": 51.4
    },
    {
     "pair": "TENSION~RESOLUTION",
     "target14": 47.0,
     "measured_L14": 3.0,
     "abs_err": 43.9
    },
    {
     "pair": "RETRIEVAL~CONSTRUCTION",
     "target14": 52.9,
     "measured_L14": 3.0,
     "abs_err": 49.8
    },
    {
     "pair": "FAMILIARITY~NOVELTY",
     "target14": 52.5,
     "measured_L14": 2.3,
     "abs_err": 50.2
    },
    {
     "pair": "CONFABULATION~CALIBRATION",
     "target14": 49.2,
     "measured_L14": 2.8,
     "abs_err": 46.5
    },
    {
     "pair": "SATURATION~LIMIT",
     "target14": 56.

shipped to gdrive:semcore/e6/full_20260822_0121
